# 🎯 [K14_M08] 원인 가설과 개선 제안 만들기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K14/k14_m08_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·부대관리 (Level 1) / K14 군 생활 데이터 분석 |
| **미션** | 미션 08. 원인 가설과 개선 제안 만들기 (소요시간: 약 1.5시간) |
| **실습 유형** | 군 생활 데이터 분석, HITL 수기 검산, 시각화 왜곡 방지 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [Colab 환경 설정 및 헬퍼 함수] 데이터 분석, 가상 데이터 로더 & 마크다운 렌더러
# ============================================================================
!pip install -q pandas numpy tabulate matplotlib seaborn

import pandas as pd
import numpy as np
import json
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# 폰트 설정 (Colab / Linux 호환)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ----------------------------------------------------------------------------
# [데이터셋 자동 로드] 가상 한빛대대 4주간 생활관 생활 데이터 (12행)
# ----------------------------------------------------------------------------
data_records = [
    {'주차': '1주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 112, '싸지방': 54, 'PX': 140, '수면만족': 3.8, '식단만족': 3.9, '건의': 3},
    {'주차': '1주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 98, '싸지방': 31, 'PX': 160, '수면만족': 3.6, '식단만족': 3.5, '건의': 4},
    {'주차': '1주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 104, '싸지방': 47, 'PX': 118, '수면만족': 3.7, '식단만족': 3.2, '건의': 5},
    {'주차': '2주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 118, '싸지방': 58, 'PX': 138, '수면만족': 3.8, '식단만족': 3.8, '건의': 2},
    {'주차': '2주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 92, '싸지방': 29, 'PX': 166, '수면만족': 3.4, '식단만족': 3.4, '건의': 5},
    {'주차': '2주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 106, '싸지방': 50, 'PX': 121, '수면만족': 3.6, '식단만족': 3.1, '건의': 7},
    {'주차': '3주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 121, '싸지방': 61, 'PX': 142, '수면만족': 3.7, '식단만족': 4.0, '건의': 4},
    {'주차': '3주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 85, '싸지방': 26, 'PX': 171, '수면만족': 3.2, '식단만족': 3.4, '건의': 7},
    {'주차': '3주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 103, '싸지방': 49, 'PX': 117, '수면만족': 3.7, '식단만족': 3.3, '건의': 9},
    {'주차': '4주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 125, '싸지방': 66, 'PX': 135, '수면만족': 3.7, '식단만족': 4.1, '건의': 3},
    {'주차': '4주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 80, '싸지방': 24, 'PX': 178, '수면만족': 3.0, '식단만족': 3.3, '건의': 8},
    {'주차': '4주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 108, '싸지방': 52, 'PX': 120, '수면만족': 3.7, '식단만족': 3.2, '건의': 12}
]

df = pd.DataFrame(data_records)

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드합니다."""
    filename = f"k14_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K14", "../uploads/K14"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 분석 결과 및 검증 리포트를 시각적으로 렌더링합니다."""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 분석 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ [K14] 실습 환경, 12행 생활관 데이터셋(df) 및 시각화 렌더러 준비 완료!")
display(df)



---

## 🎯 미션 08. 원인 가설과 개선 제안 만들기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **원인 가설과 개선 제안을 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **주목할 항목마다 데이터와 특이사항 메모를 근거로 원인 가설을 AI와 함께 2~3개 세우고, 각 가설을 확인하려면 어떤 추가 자료가 필요한지 적습니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `원인 가설 표, 추가 확인 자료, 개선 제안 우선순위` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 8 원인 가설 표

| **주목 항목** | **가설** | **근거(데이터·메모)** | **추가로 확인할 자료** | **개선 제안** | **우선순위** |
| --- | --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 가설·제안 도출 — 근거 있는 가설과 확인 방법, 현실적인 개선 제안을 정리했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(원인 가설 표, 추가 확인 자료, 개선 제안 우선순위)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 08.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_08_prompt = """[다각도 원인 가설 수립 및 개선 제안 우선순위 도출 질의]
당신은 병영생활 개선 정책 연구관입니다. 앞서 식별한 이상 징후(3중대 건의 급증 33건, 2중대 수면 만족도 급감 및 체련실 이용 저조)에 대해, 가상 데이터셋의 '건의 유형별 통계'와 '주차별 특이사항 메모'를 연계하여 다각도의 원인 가설(2~3개)을 수립해 주십시오.
각 가설을 지지하는 정황 증거와 단정 방지 문구를 포함하고, 가설 검증을 위해 추가로 확인해야 할 부대 자료 목록 및 긴급성/실현성을 고려한 개선 제안 우선순위표를 도출해 주십시오."""

mission_08_result = """## 1. 원인 가설 수립표
| 주목 지표 현상 | 다각도 원인 가설 (2~3개) | 가설 지지 정황 단서 (특이사항 메모) |
| :--- | :--- | :--- |
| **3중대 건의 급증 (4주 33건)** | 1. 생활관 난방기 이상 소음으로 인한 수면 방해 가능성<br>2. 3중대 건의함 접근성 개선 또는 분대장 적극 수렴<br>3. 환절기 온습도 불균형에 따른 시설 불만 | - 1주차: 3중대 난방기 이상 소음 접수(점검 전)<br>- 4주차: 3중대 난방기 점검 예정 공지<br>- 건의 유형 중 '시설·환경'이 17건으로 과반 |
| **2중대 체력단련실 이용 급감** | 1. 2주차 야간 경계 근무 교대 시간 조정으로 인한 피로 누적<br>2. 특정 운동 기구 고장 또는 정비 소요 발생 가능성 | - 2주차 메모: 야간 근무 교대 시간 조정됨<br>- 건의 유형 중 '일과·시간' 건의가 10건으로 최다 |

## 2. 추가 확인 자료 목록
1. **대대 시설관리반 정비 일지**: 3중대 난방기 실제 수리 일자 및 부품 교체 내역 확인
2. **2중대 일과운영표 및 근무 편성표**: 야간 경계근무 변경 전후의 수면 및 체력단련 가능 시간대 비교

## 3. 개선 제안 우선순위표
| 우선순위 | 구분 | 개선 제안 과제 | 예상 소요 및 기대 효과 |
| :--- | :--- | :--- | :--- |
| **1순위 (긴급)** | 시설 정비 | 3중대 난방기 점검 조기 완료 및 소음 방지 조치 | 즉각적인 건의 감소 및 수면 환경 안정화 |
| **2순위 (단기)** | 일과 조정 | 2중대 야간근무자 전용 체력단련 및 휴식 시간대 지정 | 체력단련실 이용 회복 및 피로도 개선 |
| **3순위 (중장기)** | 제도 개선 | 중대별 건의함 온라인화 및 주간 처리 결과 게시판 공지 | 장병 소통 만족도 제고 및 신뢰 증진 |

*안내: 본 개선 제안은 가상 데이터 분석에 근거한 건의안이며, 최종 실행 여부는 부대 지휘계통 결심에 따름.*

## 4. 도전 과제: 대안 가설 및 확증 편향 점검
- **대안 가설**: 
- **비판적 검토 결과**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 08. 원인 가설과 개선 제안 만들기", mission_08_prompt, mission_08_result)

